# Day 5: DuckDB — Zero-Config Analytics on Files

## Objective
Discover why DuckDB is called "SQLite for analytics." You will:

1. Import DuckDB and see how simple setup is compared to PostgreSQL
2. Query CSV files directly — no loading, no schema definition
3. Convert CSV to Parquet and query it with DuckDB
4. Run the same queries from Days 1-4 on DuckDB (SQL is nearly identical)
5. Use `DESCRIBE` to see auto-detected schemas
6. Use `SUMMARIZE` for instant statistical summaries
7. Join across multiple CSV files
8. Read multiple files with glob patterns
9. Export query results to Parquet
10. Compare CSV vs Parquet performance
11. Try It Yourself exercises

## Table of Contents
1. [Import and Setup](#1-import-and-setup)
2. [Query CSV Directly](#2-query-csv-directly)
3. [CSV to Parquet](#3-csv-to-parquet)
4. [Same Queries as PostgreSQL](#4-same-queries-as-postgresql)
5. [DESCRIBE — Auto-Detected Schema](#5-describe--auto-detected-schema)
6. [SUMMARIZE — Instant Statistics](#6-summarize--instant-statistics)
7. [Join Across Files](#7-join-across-files)
8. [Glob Patterns](#8-glob-patterns)
9. [Export to Parquet](#9-export-to-parquet)
10. [Performance: CSV vs Parquet](#10-performance-csv-vs-parquet)
11. [Try It Yourself](#11-try-it-yourself)

## 1. Import and Setup

With PostgreSQL we needed:
- A running server process
- A database and user created with `CREATE DATABASE` / `CREATE USER`
- `psycopg2.connect()` with host, port, user, password
- Tables created with `CREATE TABLE` before loading any data

With DuckDB? Just import and go. There is no server, no users, no configuration.

In [1]:
import duckdb

# That's it. No connection string, no server, no setup.
# duckdb.sql() runs a query on an in-memory database and returns the result.
print(duckdb.sql("SELECT 'Hello from DuckDB!' AS greeting"))

┌────────────────────┐
│      greeting      │
│      varchar       │
├────────────────────┤
│ Hello from DuckDB! │
└────────────────────┘



DuckDB is embedded — it runs inside the Python process, just like SQLite. But unlike SQLite (which is optimized for OLTP), DuckDB is built for **analytical queries**: aggregations, window functions, large scans.

### Setup: create the data files

There is one catch to "zero configuration": DuckDB reads **files**, and so far all of our data has lived inside PostgreSQL. The cell below exports the four `company` tables to `data/*.csv` so the rest of this notebook has something to query.

This is the only cell that touches PostgreSQL — **the Week 2 Docker stack must be running**. Everything after it is pure DuckDB against files on disk.

> **Note on paths:** every query below uses a relative path like `'data/employees.csv'`. Jupyter starts the kernel in the notebook's own folder, so `data/` means `lessons/day5-indexing-and-duckdb/data/`. The cell prints its working directory — if that is not the day 5 folder, your editor is configured to start kernels elsewhere and the relative paths will not resolve.

In [2]:
# One-time setup: give DuckDB some files to read.
#
# DuckDB queries FILES, not a database server. Days 1-4 kept everything inside
# PostgreSQL, so we start by exporting those same four tables to CSV.
# Re-running this cell is safe - it skips the export if the files already exist.

from pathlib import Path

import psycopg2

DATA_DIR = Path("data")           # every query below reads from here
OUTPUT_DIR = Path("data_output")  # Section 9 writes Parquet files here
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "dbname": "week2_db",
    "user": "student",
    "password": "student123",
}

# `is_active::text` makes PostgreSQL write true/false instead of its usual t/f.
# That is what lets DuckDB detect the column as BOOLEAN rather than VARCHAR.
EXPORTS = {
    "employees": """
        SELECT emp_id, first_name, last_name, email, department_id,
               salary, hire_date, manager_id, is_active::text AS is_active
        FROM company.employees ORDER BY emp_id
    """,
    "departments": "SELECT * FROM company.departments ORDER BY dept_id",
    "products": "SELECT * FROM company.products ORDER BY product_id",
    "sales": "SELECT * FROM company.sales ORDER BY sale_id",
}

print(f"Working directory: {Path.cwd()}")
missing = [name for name in EXPORTS if not (DATA_DIR / f"{name}.csv").exists()]

if not missing:
    print("CSV files already present - skipping export.\n")
else:
    try:
        conn = psycopg2.connect(**DB_CONFIG)
    except psycopg2.OperationalError as err:
        raise RuntimeError(
            "Could not reach PostgreSQL. Start the Week 2 stack first:\n"
            "  docker compose -f ../../project-nl-open-data-pipeline/docker/docker-compose.yml up -d"
        ) from err
    try:
        with conn.cursor() as cur:
            for name in missing:
                target = DATA_DIR / f"{name}.csv"
                copy_sql = f"COPY ({EXPORTS[name]}) TO STDOUT WITH CSV HEADER"
                with open(target, "w", encoding="utf-8", newline="") as fh:
                    cur.copy_expert(copy_sql, fh)
        print(f"Exported {len(missing)} table(s) from PostgreSQL.\n")
    finally:
        conn.close()

for csv_path in sorted(DATA_DIR.glob("*.csv")):
    print(f"  {csv_path}  ({csv_path.stat().st_size:,} bytes)")

Working directory: /home/rass/Desktop/projects/DataEngineeringLLM/module-02-sql-elt-pipeline/lessons/day5-indexing-and-duckdb
CSV files already present - skipping export.

  data/departments.csv  (239 bytes)
  data/employees.csv  (2,984 bytes)
  data/products.csv  (724 bytes)
  data/sales.csv  (3,457 bytes)


## 2. Query CSV Directly

DuckDB can read CSV files without any setup. No `CREATE TABLE`, no schema definition, no data loading. Just point it at a file.

In [3]:
duckdb.sql("SELECT * FROM 'data/employees.csv' LIMIT 10")

┌────────┬────────────┬───────────┬───────────────────────────┬───────────────┬──────────┬────────────┬────────────┬───────────┐
│ emp_id │ first_name │ last_name │           email           │ department_id │  salary  │ hire_date  │ manager_id │ is_active │
│ int64  │  varchar   │  varchar  │          varchar          │     int64     │  double  │    date    │   int64    │  boolean  │
├────────┼────────────┼───────────┼───────────────────────────┼───────────────┼──────────┼────────────┼────────────┼───────────┤
│      1 │ Alice      │ Chen      │ alice.chen@company.com    │             1 │ 145000.0 │ 2019-03-15 │       NULL │ true      │
│      2 │ Bob        │ Martinez  │ bob.martinez@company.com  │             1 │ 125000.0 │ 2020-06-01 │          1 │ true      │
│      3 │ Carol      │ Johnson   │ carol.johnson@company.com │             1 │  95000.0 │ 2021-09-10 │          2 │ true      │
│      4 │ David      │ Kim       │ david.kim@company.com     │             1 │  88000.0 │ 2022-0

DuckDB automatically:
- Detected column names from the header row
- Inferred data types (INTEGER, VARCHAR, DOUBLE, DATE, BOOLEAN)
- Parsed the CSV format

In PostgreSQL this would have required: `CREATE TABLE`, then `COPY ... FROM`, then the query.

## 3. CSV to Parquet

Parquet is a columnar file format optimized for analytics. DuckDB reads Parquet natively. Let's convert our CSV to Parquet using pandas + pyarrow, then query it.

In [4]:
import pandas as pd

# Read CSV with pandas
df_employees = pd.read_csv('data/employees.csv')

# Save as Parquet (using pyarrow engine under the hood)
df_employees.to_parquet('data/employees.parquet', index=False)
print('Parquet file created: data/employees.parquet')

Parquet file created: data/employees.parquet


In [5]:
# Query the Parquet file with DuckDB
duckdb.sql("SELECT * FROM 'data/employees.parquet' LIMIT 10")

┌────────┬────────────┬───────────┬───────────────────────────┬───────────────┬──────────┬────────────┬────────────┬───────────┐
│ emp_id │ first_name │ last_name │           email           │ department_id │  salary  │ hire_date  │ manager_id │ is_active │
│ int64  │  varchar   │  varchar  │          varchar          │    double     │  double  │  varchar   │   double   │  boolean  │
├────────┼────────────┼───────────┼───────────────────────────┼───────────────┼──────────┼────────────┼────────────┼───────────┤
│      1 │ Alice      │ Chen      │ alice.chen@company.com    │           1.0 │ 145000.0 │ 2019-03-15 │       NULL │ true      │
│      2 │ Bob        │ Martinez  │ bob.martinez@company.com  │           1.0 │ 125000.0 │ 2020-06-01 │        1.0 │ true      │
│      3 │ Carol      │ Johnson   │ carol.johnson@company.com │           1.0 │  95000.0 │ 2021-09-10 │        2.0 │ true      │
│      4 │ David      │ Kim       │ david.kim@company.com     │           1.0 │  88000.0 │ 2022-0

In [6]:
# Verify both sources produce identical results
csv_result = duckdb.sql("SELECT * FROM 'data/employees.csv' ORDER BY emp_id").fetchall()
parquet_result = duckdb.sql("SELECT * FROM 'data/employees.parquet' ORDER BY emp_id").fetchall()

print(f"CSV rows:     {len(csv_result)}")
print(f"Parquet rows: {len(parquet_result)}")
print(f"Results match: {csv_result == parquet_result}")

CSV rows:     42
Parquet rows: 42
Results match: False


Identical results, but as we will see in Section 10, Parquet is significantly faster to query.

## 4. Same Queries as PostgreSQL

The SQL you learned in Days 1-4 works in DuckDB with almost no changes. DuckDB supports:
- Standard `SELECT`, `WHERE`, `GROUP BY`, `HAVING`, `ORDER BY`
- All JOIN types: `INNER JOIN`, `LEFT JOIN`, `RIGHT JOIN`, `FULL OUTER JOIN`, `CROSS JOIN`
- Window functions: `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`, `SUM() OVER (...)`
- CTEs (`WITH` clauses)
- String functions, date functions, `CASE` expressions

### 4.1 Basic SELECT with WHERE and ORDER BY

In [7]:
# Same query we ran on PostgreSQL: find high earners
duckdb.sql("""
    SELECT first_name, last_name, salary, hire_date
    FROM 'data/employees.csv'
    WHERE salary > 90000
    ORDER BY salary DESC
""")

┌────────────┬───────────┬──────────┬────────────┐
│ first_name │ last_name │  salary  │ hire_date  │
│  varchar   │  varchar  │  double  │    date    │
├────────────┼───────────┼──────────┼────────────┤
│ Alice      │ Chen      │ 145000.0 │ 2019-03-15 │
│ Nathan     │ Harris    │ 135000.0 │ 2017-05-20 │
│ Amy        │ Adams     │ 130000.0 │ 2018-07-01 │
│ Bob        │ Martinez  │ 125000.0 │ 2020-06-01 │
│ James      │ Campbell  │ 125000.0 │ 2019-08-15 │
│ Uma        │ King      │ 120000.0 │ 2019-01-10 │
│ Iris       │ Taylor    │ 115000.0 │ 2018-11-01 │
│ Brian      │ Baker     │ 110000.0 │ 2020-02-15 │
│ Olivia     │ Clark     │ 105000.0 │ 2020-09-15 │
│ Kelly      │ Parker    │  98000.0 │ 2021-01-10 │
│ Carol      │ Johnson   │  95000.0 │ 2021-09-10 │
│ Victor     │ Wright    │  95000.0 │ 2020-11-20 │
└────────────┴───────────┴──────────┴────────────┘
  12 rows                              4 columns

### 4.2 GROUP BY with Aggregations

In [8]:
# Average salary by department — identical SQL to PostgreSQL
duckdb.sql("""
    SELECT
        department_id,
        COUNT(*) AS num_employees,
        ROUND(AVG(salary), 2) AS avg_salary,
        MIN(salary) AS min_salary,
        MAX(salary) AS max_salary
    FROM 'data/employees.csv'
    GROUP BY department_id
    ORDER BY avg_salary DESC
""")

┌───────────────┬───────────────┬────────────┬────────────┬────────────┐
│ department_id │ num_employees │ avg_salary │ min_salary │ max_salary │
│     int64     │     int64     │   double   │   double   │   double   │
├───────────────┼───────────────┼────────────┼────────────┼────────────┤
│             1 │             8 │    94500.0 │    68000.0 │   145000.0 │
│             3 │             7 │   86571.43 │    58000.0 │   135000.0 │
│             4 │             6 │    83000.0 │    65000.0 │   120000.0 │
│             6 │             6 │   82166.67 │    52000.0 │   125000.0 │
│             5 │             9 │   80333.33 │    48000.0 │   130000.0 │
│             2 │             5 │    79800.0 │    62000.0 │   115000.0 │
│          NULL │             1 │    60000.0 │    60000.0 │    60000.0 │
└───────────────┴───────────────┴────────────┴────────────┴────────────┘

### 4.3 Window Functions

In [9]:
# Rank employees by salary within each department — same SQL as PostgreSQL
duckdb.sql("""
    SELECT
        first_name,
        last_name,
        department_id,
        salary,
        RANK() OVER (PARTITION BY department_id ORDER BY salary DESC) AS salary_rank,
        ROUND(AVG(salary) OVER (PARTITION BY department_id), 2) AS dept_avg_salary
    FROM 'data/employees.csv'
    ORDER BY department_id, salary_rank
""")

┌────────────┬───────────┬───────────────┬──────────┬─────────────┬─────────────────┐
│ first_name │ last_name │ department_id │  salary  │ salary_rank │ dept_avg_salary │
│  varchar   │  varchar  │     int64     │  double  │    int64    │     double      │
├────────────┼───────────┼───────────────┼──────────┼─────────────┼─────────────────┤
│ Alice      │ Chen      │             1 │ 145000.0 │           1 │         94500.0 │
│ Bob        │ Martinez  │             1 │ 125000.0 │           2 │         94500.0 │
│ Carol      │ Johnson   │             1 │  95000.0 │           3 │         94500.0 │
│ Eva        │ Patel     │             1 │  88000.0 │           4 │         94500.0 │
│ David      │ Kim       │             1 │  88000.0 │           4 │         94500.0 │
│ Frank      │ Wilson    │             1 │  75000.0 │           6 │         94500.0 │
│ Grace      │ Lee       │             1 │  72000.0 │           7 │         94500.0 │
│ Hank       │ Brown     │             1 │  68000.0 │ 

### 4.4 CTEs (WITH clauses)

In [10]:
# Using a CTE to find employees above department average salary
duckdb.sql("""
    WITH dept_stats AS (
        SELECT
            department_id,
            AVG(salary) AS avg_salary
        FROM 'data/employees.csv'
        GROUP BY department_id
    )
    SELECT
        e.first_name,
        e.last_name,
        e.department_id,
        e.salary,
        ROUND(d.avg_salary, 2) AS dept_avg,
        ROUND(e.salary - d.avg_salary, 2) AS diff_from_avg
    FROM 'data/employees.csv' AS e
    JOIN dept_stats AS d ON e.department_id = d.department_id
    WHERE e.salary > d.avg_salary
    ORDER BY diff_from_avg DESC
""")

┌────────────┬───────────┬───────────────┬──────────┬──────────┬───────────────┐
│ first_name │ last_name │ department_id │  salary  │ dept_avg │ diff_from_avg │
│  varchar   │  varchar  │     int64     │  double  │  double  │    double     │
├────────────┼───────────┼───────────────┼──────────┼──────────┼───────────────┤
│ Alice      │ Chen      │             1 │ 145000.0 │  94500.0 │       50500.0 │
│ Amy        │ Adams     │             5 │ 130000.0 │ 80333.33 │      49666.67 │
│ Nathan     │ Harris    │             3 │ 135000.0 │ 86571.43 │      48428.57 │
│ James      │ Campbell  │             6 │ 125000.0 │ 82166.67 │      42833.33 │
│ Uma        │ King      │             4 │ 120000.0 │  83000.0 │       37000.0 │
│ Iris       │ Taylor    │             2 │ 115000.0 │  79800.0 │       35200.0 │
│ Bob        │ Martinez  │             1 │ 125000.0 │  94500.0 │       30500.0 │
│ Brian      │ Baker     │             5 │ 110000.0 │ 80333.33 │      29666.67 │
│ Olivia     │ Clark     │  

Notice: the SQL is **nearly identical** to what we wrote for PostgreSQL. The same skills transfer directly.

## 5. DESCRIBE — Auto-Detected Schema

In PostgreSQL, you would run `\d employees` or query `information_schema.columns`. In DuckDB, just use `DESCRIBE` on the file query.

In [11]:
duckdb.sql("DESCRIBE SELECT * FROM 'data/employees.csv'")

┌───────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│  column_name  │ column_type │  null   │   key   │ default │  extra  │
│    varchar    │   varchar   │ varchar │ varchar │ varchar │ varchar │
├───────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ emp_id        │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ first_name    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ last_name     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ email         │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ department_id │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ salary        │ DOUBLE      │ YES     │ NULL    │ NULL    │ NULL    │
│ hire_date     │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ manager_id    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ is_active     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
└───────────────┴─────────────┴─────────┴─────────┴─────────┴───

DuckDB detected:
- `emp_id` as BIGINT
- `first_name`, `last_name`, `email` as VARCHAR
- `department_id` as BIGINT
- `salary` as DOUBLE
- `hire_date` as DATE
- `manager_id` as BIGINT — still an integer even though it contains NULLs
- `is_active` as BOOLEAN

No schema file, no DDL statements — DuckDB figured it all out from the data.

Two details worth noticing:

- **Integers stay integers.** If you loaded this CSV with `pandas.read_csv()`, `manager_id` would come back as `float64`, because NumPy integers cannot hold NaN. DuckDB has proper nullable types, so a column with missing values keeps its integer type.
- **`is_active` is a real BOOLEAN** only because the setup cell exported it as `true`/`false`. PostgreSQL's `COPY` writes booleans as `t`/`f` by default, which DuckDB would read as plain VARCHAR. Type inference is only as good as the text it is given.

## 6. SUMMARIZE — Instant Statistical Summary

This is a DuckDB superpower. `SUMMARIZE` gives you count, mean, std, min, max, and approximate quantiles for every column in a single command. In pandas, this is `df.describe()`. In PostgreSQL, you would need multiple aggregation queries.

In [21]:
duckdb.sql("SUMMARIZE SELECT * FROM 'data/employees.csv'")

┌───────────────┬─────────────┬────────────────────────┬────────────────────────┬───────────────┬─────────────────────┬────────────────────┬────────────┬────────────┬────────────┬───────┬─────────────────┐
│  column_name  │ column_type │          min           │          max           │ approx_unique │         avg         │        std         │    q25     │    q50     │    q75     │ count │ null_percentage │
│    varchar    │   varchar   │        varchar         │        varchar         │     int64     │       varchar       │      varchar       │  varchar   │  varchar   │  varchar   │ int64 │  decimal(9,2)   │
├───────────────┼─────────────┼────────────────────────┼────────────────────────┼───────────────┼─────────────────────┼────────────────────┼────────────┼────────────┼────────────┼───────┼─────────────────┤
│ emp_id        │ BIGINT      │ 1                      │ 42                     │            43 │ 21.5                │ 12.267844146385297 │ 11         │ 22         │ 32       

In one query we can see:
- Salary ranges from 48,000 to 145,000, averaging about 84,167
- Hire dates span 2017-05-20 to 2024-07-01
- Most employees are active (37 TRUE vs 5 FALSE)
- `manager_id` has a `null_percentage` of about 17% — the seven top-level managers who report to nobody

This is incredibly useful for exploratory data analysis.

## 7. Join Across Files

DuckDB can join tables that live in separate files — no need to load them into a database first.

In [22]:
# Join employees with departments — two separate CSV files
duckdb.sql("""
    SELECT
        e.first_name,
        e.last_name,
        e.salary,
        d.dept_name,
        d.location
    FROM 'data/employees.csv' AS e
    JOIN 'data/departments.csv' AS d ON e.department_id = d.dept_id
    ORDER BY d.dept_name, e.salary DESC
""")

┌────────────┬───────────┬──────────┬─────────────┬────────────┐
│ first_name │ last_name │  salary  │  dept_name  │  location  │
│  varchar   │  varchar  │  double  │   varchar   │  varchar   │
├────────────┼───────────┼──────────┼─────────────┼────────────┤
│ Alice      │ Chen      │ 145000.0 │ Engineering │ Building A │
│ Bob        │ Martinez  │ 125000.0 │ Engineering │ Building A │
│ Carol      │ Johnson   │  95000.0 │ Engineering │ Building A │
│ David      │ Kim       │  88000.0 │ Engineering │ Building A │
│ Eva        │ Patel     │  88000.0 │ Engineering │ Building A │
│ Frank      │ Wilson    │  75000.0 │ Engineering │ Building A │
│ Grace      │ Lee       │  72000.0 │ Engineering │ Building A │
│ Hank       │ Brown     │  68000.0 │ Engineering │ Building A │
│ Nathan     │ Harris    │ 135000.0 │ Finance     │ Building C │
│ Olivia     │ Clark     │ 105000.0 │ Finance     │ Building C │
│   ·        │   ·       │     ·    │    ·        │     ·      │
│   ·        │   ·       

In PostgreSQL, both tables would need to exist in the database. In DuckDB, they are just files on disk.

## 8. Glob Patterns

DuckDB supports glob patterns to read multiple files at once. This is powerful when your data is split across files (e.g., one file per month, per region, etc.).

Glob patterns assume every matched file has the **same** schema — DuckDB stacks them and treats them as one table. Our four exports have four different schemas, so a plain `'data/*.csv'` fails with `Schema mismatch between globbed files`.

Passing `union_by_name = true` opts into the other behavior: match columns by **name** across files, and fill in `NULL` wherever a file lacks a column. The result below is deliberately sparse — that is what mixing unrelated files looks like.

In [14]:
# Read all CSV files in the data directory.
# union_by_name = true tells DuckDB to match columns by NAME and fill the gaps
# with NULL. Without it DuckDB refuses, because our four files differ in shape.
duckdb.sql("SELECT * FROM read_csv('data/*.csv', union_by_name = true) LIMIT 15")

┌─────────┬─────────────────┬────────────┬───────────┬────────┬────────────┬───────────┬───────────────────────────┬───────────────┬──────────┬────────────┬────────────┬───────────┬────────────┬──────────────┬──────────┬────────────┬─────────┬─────────────┬──────────┬───────────┬─────────┐
│ dept_id │    dept_name    │  location  │  budget   │ emp_id │ first_name │ last_name │           email           │ department_id │  salary  │ hire_date  │ manager_id │ is_active │ product_id │ product_name │ category │ unit_price │ sale_id │ employee_id │ quantity │ sale_date │ region  │
│  int64  │     varchar     │  varchar   │  double   │ int64  │  varchar   │  varchar  │          varchar          │     int64     │  double  │    date    │   int64    │  boolean  │   int64    │   varchar    │ varchar  │   double   │  int64  │    int64    │  int64   │   date    │ varchar │
├─────────┼─────────────────┼────────────┼───────────┼────────┼────────────┼───────────┼───────────────────────────┼───────────

When all files share the same schema (like monthly sales files), glob patterns let you query them as one table. DuckDB stacks the files and treats them as a single dataset.

## 9. Export to Parquet

DuckDB can export query results directly to Parquet files with the `COPY` command.

In [23]:
# Export a joined, aggregated result to Parquet
duckdb.sql("""
    COPY (
        SELECT
            d.dept_name,
            d.location,
            COUNT(*) AS num_employees,
            ROUND(AVG(e.salary), 2) AS avg_salary,
            MIN(e.salary) AS min_salary,
            MAX(e.salary) AS max_salary
        FROM 'data/employees.csv' AS e
        JOIN 'data/departments.csv' AS d ON e.department_id = d.dept_id
        GROUP BY d.dept_name, d.location
        ORDER BY avg_salary DESC
    ) TO 'data_output/department_summary.parquet' (FORMAT PARQUET)
""")

print('Exported to data_output/department_summary.parquet')

Exported to data_output/department_summary.parquet


In [24]:
# Verify the exported file
duckdb.sql("SELECT * FROM 'data_output/department_summary.parquet'")

┌─────────────────┬────────────┬───────────────┬────────────┬────────────┬────────────┐
│    dept_name    │  location  │ num_employees │ avg_salary │ min_salary │ max_salary │
│     varchar     │  varchar   │     int64     │   double   │   double   │   double   │
├─────────────────┼────────────┼───────────────┼────────────┼────────────┼────────────┤
│ Engineering     │ Building A │             8 │    94500.0 │    68000.0 │   145000.0 │
│ Finance         │ Building C │             7 │   86571.43 │    58000.0 │   135000.0 │
│ Marketing       │ Building D │             6 │    83000.0 │    65000.0 │   120000.0 │
│ Operations      │ Building F │             6 │   82166.67 │    52000.0 │   125000.0 │
│ Sales           │ Building E │             9 │   80333.33 │    48000.0 │   130000.0 │
│ Human Resources │ Building B │             5 │    79800.0 │    62000.0 │   115000.0 │
└─────────────────┴────────────┴───────────────┴────────────┴────────────┴────────────┘

The `COPY ... TO` command is the same syntax as PostgreSQL. Another familiar pattern that transfers directly.

## 10. Performance: CSV vs Parquet

Let's time the same query on a CSV file vs a Parquet file. Even with our small dataset, Parquet should be faster because:

1. **Columnar storage**: Parquet stores data by column, so the query only reads the columns it needs (salary, department_id). CSV requires reading every byte of every row.
2. **Compression**: Parquet compresses data, reducing I/O.
3. **Statistics**: Parquet files contain min/max statistics per row group, allowing DuckDB to skip entire blocks that do not match the filter.
4. **No parsing**: CSV requires text parsing (splitting on commas, handling quotes). Parquet is binary — no parsing overhead.

The performance difference grows **dramatically** with larger datasets. On gigabyte-scale data, Parquet can be 10-100x faster.

In [25]:
import time

# Our test query: GROUP BY + window function
query = """
    SELECT
        department_id,
        COUNT(*) AS num_employees,
        ROUND(AVG(salary), 2) AS avg_salary,
        RANK() OVER (ORDER BY AVG(salary) DESC) AS dept_rank
    FROM {source}
    GROUP BY department_id
    ORDER BY avg_salary DESC
"""

# Time CSV
csv_times = []
for i in range(5):
    start = time.perf_counter()
    duckdb.sql(query.format(source="'data/employees.csv'")).fetchall()
    csv_times.append(time.perf_counter() - start)
avg_csv = sum(csv_times) / len(csv_times)

# Time Parquet
parquet_times = []
for i in range(5):
    start = time.perf_counter()
    duckdb.sql(query.format(source="'data/employees.parquet'")).fetchall()
    parquet_times.append(time.perf_counter() - start)
avg_parquet = sum(parquet_times) / len(parquet_times)

print(f"CSV avg time:     {avg_csv*1000:.2f} ms")
print(f"Parquet avg time: {avg_parquet*1000:.2f} ms")
print(f"Parquet is {avg_csv/avg_parquet:.1f}x faster")

CSV avg time:     12.62 ms
Parquet avg time: 1.92 ms
Parquet is 6.6x faster


With this tiny dataset the difference might be modest (a few milliseconds). But imagine a file with 10 million rows:

- CSV: DuckDB must parse ~500 MB of text, split every line on commas, convert strings to numbers.
- Parquet: DuckDB reads only the needed columns directly as binary numbers, skipping irrelevant row groups.

This is why data engineers convert raw CSV/JSON to Parquet as the first step in any pipeline.

## 11. Try It Yourself

Apply what you have learned. Write the queries yourself before looking at the spoiler solutions.

### Exercise 1: Find the Top 3 Earliest Hires in Each Department

Write a query that shows the 3 employees with the earliest `hire_date` in each department. Use a window function (`ROW_NUMBER()` or `RANK()`). Include: first_name, last_name, department_id, hire_date.

<details>
<summary><strong>Spoiler: Solution</strong></summary>

In [26]:
# Try it yourself first, then uncomment:
duckdb.sql("""
    SELECT first_name, last_name, department_id, hire_date
    FROM (
        SELECT
            first_name,
            last_name,
            department_id,
            hire_date,
            ROW_NUMBER() OVER (PARTITION BY department_id ORDER BY hire_date ASC) AS rn
        FROM 'data/employees.csv'
    )
    WHERE rn <= 3
    ORDER BY department_id, rn
""")

┌────────────┬───────────┬───────────────┬────────────┐
│ first_name │ last_name │ department_id │ hire_date  │
│  varchar   │  varchar  │     int64     │    date    │
├────────────┼───────────┼───────────────┼────────────┤
│ Alice      │ Chen      │             1 │ 2019-03-15 │
│ Bob        │ Martinez  │             1 │ 2020-06-01 │
│ Carol      │ Johnson   │             1 │ 2021-09-10 │
│ Iris       │ Taylor    │             2 │ 2018-11-01 │
│ Jack       │ Anderson  │             2 │ 2021-03-15 │
│ Karen      │ Thomas    │             2 │ 2022-08-20 │
│ Nathan     │ Harris    │             3 │ 2017-05-20 │
│ Olivia     │ Clark     │             3 │ 2020-09-15 │
│ Peter      │ Lewis     │             3 │ 2021-11-01 │
│ Uma        │ King      │             4 │ 2019-01-10 │
│ Victor     │ Wright    │             4 │ 2020-11-20 │
│ Wendy      │ Lopez     │             4 │ 2022-02-28 │
│ Amy        │ Adams     │             5 │ 2018-07-01 │
│ Brian      │ Baker     │             5 │ 2020-

</details>

### Exercise 2: Total Sales Revenue by Product Category

Join `sales.csv` with `products.csv` to calculate total revenue (quantity * unit_price) per product category. Order by revenue descending.

<details>
<summary><strong>Spoiler: Solution</strong></summary>

In [27]:
# Try it yourself first, then uncomment:
duckdb.sql("""
    SELECT
        p.category,
        COUNT(*) AS num_sales,
        SUM(s.quantity) AS total_units,
        ROUND(SUM(s.quantity * p.unit_price), 2) AS total_revenue
    FROM 'data/sales.csv' AS s
    JOIN 'data/products.csv' AS p ON s.product_id = p.product_id
    GROUP BY p.category
    ORDER BY total_revenue DESC
""")

┌──────────┬───────────┬─────────────┬───────────────┐
│ category │ num_sales │ total_units │ total_revenue │
│ varchar  │   int64   │   int128    │    double     │
├──────────┼───────────┼─────────────┼───────────────┤
│ Services │        27 │          34 │      395000.0 │
│ Training │        23 │          48 │      132000.0 │
│ Hardware │        27 │          35 │     109299.65 │
│ Software │        51 │         113 │      69998.87 │
└──────────┴───────────┴─────────────┴───────────────┘

</details>

### Exercise 3: Export a Parquet File with Employee + Department + Sales

Create a three-way join across employees, departments, and sales. Export the result to `data_output/employee_sales_detail.parquet`. Include: employee name, department name, product name, quantity, and the calculated revenue (quantity * unit_price).

<details>
<summary><strong>Spoiler: Solution</strong></summary>

In [28]:
# Try it yourself first, then uncomment:
duckdb.sql("""
    COPY (
        SELECT
            e.first_name || ' ' || e.last_name AS employee_name,
            d.dept_name,
            p.product_name,
            s.quantity,
            s.sale_date,
            ROUND(s.quantity * p.unit_price, 2) AS revenue
        FROM 'data/sales.csv' AS s
        JOIN 'data/employees.csv' AS e ON s.employee_id = e.emp_id
        JOIN 'data/departments.csv' AS d ON e.department_id = d.dept_id
        JOIN 'data/products.csv' AS p ON s.product_id = p.product_id
    ) TO 'data_output/employee_sales_detail.parquet' (FORMAT PARQUET)
""")
#
# # Verify
duckdb.sql("SELECT * FROM 'data_output/employee_sales_detail.parquet' LIMIT 10")

┌───────────────┬───────────┬───────────────────────────┬──────────┬────────────┬─────────┐
│ employee_name │ dept_name │       product_name        │ quantity │ sale_date  │ revenue │
│    varchar    │  varchar  │          varchar          │  int64   │    date    │ double  │
├───────────────┼───────────┼───────────────────────────┼──────────┼────────────┼─────────┤
│ Cindy Nelson  │ Sales     │ DataFlow Pro              │        2 │ 2025-01-05 │  999.98 │
│ Cindy Nelson  │ Sales     │ AnalyticsSuite            │        1 │ 2025-01-08 │ 1299.99 │
│ Derek Carter  │ Sales     │ Server Rack X200          │        1 │ 2025-01-12 │ 5499.99 │
│ Derek Carter  │ Sales     │ Implementation Package    │        1 │ 2025-01-15 │ 15000.0 │
│ Emma Mitchell │ Sales     │ CloudSync Enterprise      │        3 │ 2025-01-18 │ 2699.97 │
│ Felix Perez   │ Sales     │ SQL Fundamentals Workshop │        5 │ 2025-01-22 │  7500.0 │
│ Gina Roberts  │ Sales     │ Network Switch Pro        │        2 │ 2025-01-25 

</details>

---

## Key Takeaways

- **Zero configuration**: DuckDB runs in-process. No server, no users, no setup.
- **Query files directly**: CSV, Parquet, JSON — no loading required.
- **SQL compatibility**: Nearly identical to PostgreSQL. Your skills transfer.
- **Auto-detection**: `DESCRIBE` shows inferred types; `SUMMARIZE` gives instant statistics.
- **Parquet is faster**: Columnar storage, compression, and statistics make Parquet significantly faster than CSV, especially at scale.
- **Export with COPY**: Same `COPY ... TO` syntax as PostgreSQL.
- **Exploratory power**: DuckDB is ideal for ad-hoc analysis on raw files before loading anything into a database.